In [2]:
print("hello")

hello


In [3]:
# ============================================
# 02_eda_exploration.ipynb
# Phase 2: Exploratory Data Analysis
#
# Goal: Answer real questions from our data
# Tools: pandas only (no ML, no NLP yet)
# ============================================

import sys
import os
import pandas as pd
import numpy as np

# Add project root to path
sys.path.append(os.path.join(os.getcwd(), ".."))

from src.parser       import parse_whatsapp_chat
from src.preprocessor import preprocess

# Load and preprocess data
df_raw = parse_whatsapp_chat("../data/sample/sample_chat.txt")
df     = preprocess(df_raw)

print("✅ Data loaded successfully!")
print(f"   Shape : {df.shape}")
print(f"   Dates : {df['date'].iloc[0]} → {df['date'].iloc[-1]}")
print(f"   Users : {list(df['sender'].unique())}")

✅ Data loaded successfully!
   Shape : (25, 19)
   Dates : 3/14/23 → 3/17/23
   Users : ['System', 'Alice', 'Bob', 'Charlie', 'Dave']


In [4]:
# ============================================
# CATEGORY 1: Basic Statistics
# These are the HEADLINE numbers of your app
# Every dashboard shows these first
# ============================================

print("=" * 50)
print("   CHAT OVERVIEW")
print("=" * 50)

# Total messages (all types)
total_messages = len(df)

# Only user messages (exclude system)
user_messages = len(df[df['message_type'] == 'normal'])

# Total participants
total_users = df[df['sender'] != 'System']['sender'].nunique()

# Date range
first_date = df['datetime'].min()
last_date  = df['datetime'].max()
duration   = (last_date - first_date).days

# Total words
total_words = df['word_count'].sum()

# Total emojis
total_emojis = df['emoji_count'].sum()

# Total media
total_media = df['is_media'].sum()

# Total links
total_links = df['has_url'].sum()

print(f"  Total Messages    : {total_messages}")
print(f"  User Messages     : {user_messages}")
print(f"  System Messages   : {total_messages - user_messages}")
print(f"  Total Participants: {total_users}")
print(f"  Chat Duration     : {duration} days")
print(f"  First Message     : {first_date.strftime('%B %d, %Y')}")
print(f"  Last Message      : {last_date.strftime('%B %d, %Y')}")
print(f"  Total Words       : {total_words}")
print(f"  Total Emojis      : {total_emojis}")
print(f"  Total Media Shared: {total_media}")
print(f"  Total Links Shared: {total_links}")
print("=" * 50)

   CHAT OVERVIEW
  Total Messages    : 25
  User Messages     : 19
  System Messages   : 6
  Total Participants: 4
  Chat Duration     : 3 days
  First Message     : March 14, 2023
  Last Message      : March 17, 2023
  Total Words       : 82
  Total Emojis      : 13
  Total Media Shared: 3
  Total Links Shared: 1


In [5]:
# ============================================
# CATEGORY 1 continued: Messages per user
#
# Key concept: groupby + aggregation
# groupby('sender') → groups rows by sender
# agg() → applies multiple functions at once
# ============================================

print("USER ACTIVITY ANALYSIS")
print("=" * 55)

# Filter out system messages for user analysis
users_df = df[df['sender'] != 'System'].copy()

# Aggregate multiple stats per user at once
user_stats = users_df.groupby('sender').agg(
    total_messages = ('message',      'count'),
    total_words    = ('word_count',   'sum'),
    total_emojis   = ('emoji_count',  'sum'),
    total_media    = ('is_media',     'sum'),
    total_links    = ('has_url',      'sum'),
    avg_msg_length = ('message_length','mean'),
).reset_index()

# Add percentage column
user_stats['message_percent'] = (
    user_stats['total_messages'] / 
    user_stats['total_messages'].sum() * 100
).round(1)

# Sort by most active
user_stats = user_stats.sort_values(
    'total_messages', ascending=False
)

print(user_stats.to_string(index=False))

print()
print(f"Most active user  : "
      f"{user_stats.iloc[0]['sender']} "
      f"({user_stats.iloc[0]['total_messages']} messages, "
      f"{user_stats.iloc[0]['message_percent']}%)")

print(f"Most words written: "
      f"{user_stats.nlargest(1,'total_words').iloc[0]['sender']}")

print(f"Most emojis used  : "
      f"{user_stats.nlargest(1,'total_emojis').iloc[0]['sender']}")

USER ACTIVITY ANALYSIS
 sender  total_messages  total_words  total_emojis  total_media  total_links  avg_msg_length  message_percent
  Alice               8           46             5            0            0       32.125000             34.8
    Bob               6           20             4            1            0       15.833333             26.1
Charlie               5            5             1            2            1        9.200000             21.7
   Dave               4           11             3            0            0       12.500000             17.4

Most active user  : Alice (8 messages, 34.8%)
Most words written: Alice
Most emojis used  : Alice


In [6]:
# ============================================
# CATEGORY 2: Time Analysis
#
# Key concept: value_counts()
# Counts frequency of each unique value
# Perfect for finding "most common" anything
# ============================================

print("TIME ANALYSIS")
print("=" * 55)

# ── Hourly activity ────────────────────────
print("\n[1] MESSAGES BY HOUR:")
hourly = df.groupby('hour')['message'].count()
peak_hour = hourly.idxmax()
print(f"    Peak hour: {peak_hour}:00 "
      f"({hourly.max()} messages)")
print()

for hour, count in hourly.items():
    bar = "█" * count
    print(f"    {hour:02d}:00  {bar} ({count})")

# ── Day of week activity ───────────────────
print("\n[2] MESSAGES BY DAY OF WEEK:")
day_order = ['Monday','Tuesday','Wednesday',
             'Thursday','Friday','Saturday','Sunday']

daily = df.groupby('weekday')['message'].count()
daily = daily.reindex(day_order, fill_value=0)
peak_day = daily.idxmax()

print(f"    Peak day: {peak_day} "
      f"({daily.max()} messages)")
print()

for day, count in daily.items():
    bar = "█" * count
    print(f"    {day:<12} {bar} ({count})")

# ── Monthly activity ───────────────────────
print("\n[3] MESSAGES BY MONTH:")
monthly = df.groupby('month')['message'].count()
print(monthly.to_string())

TIME ANALYSIS

[1] MESSAGES BY HOUR:
    Peak hour: 9:00 (11 messages)

    08:00  ██ (2)
    09:00  ███████████ (11)
    10:00  █████ (5)
    11:00  ████ (4)
    19:00  ███ (3)

[2] MESSAGES BY DAY OF WEEK:
    Peak day: Tuesday (13 messages)

    Monday        (0)
    Tuesday      █████████████ (13)
    Wednesday    ███ (3)
    Thursday     ███ (3)
    Friday       ██████ (6)
    Saturday      (0)
    Sunday        (0)

[3] MESSAGES BY MONTH:
month
3    25


In [7]:
# ============================================
# CATEGORY 3: Content Analysis
#
# Key concept: Word frequency
# We count how often each word appears
# across all messages
# ============================================

from collections import Counter

print("CONTENT ANALYSIS")
print("=" * 55)

# ── Word frequency ─────────────────────────
print("\n[1] TOP 15 MOST USED WORDS:")

# Only use normal messages
normal_messages = df[df['message_type'] == 'normal']['message']

# Combine all messages into one big string
all_text = " ".join(normal_messages.astype(str))

# Split into individual words
all_words = all_text.lower().split()

# Remove very short words (less than 3 characters)
# These are usually not meaningful: "a", "is", "to"
meaningful_words = [w for w in all_words if len(w) > 2]

# Count frequency
word_freq = Counter(meaningful_words)
top_words = word_freq.most_common(15)

for word, count in top_words:
    bar = "█" * count
    print(f"    {word:<15} {bar} ({count})")

# ── Emoji frequency ────────────────────────
print("\n[2] EMOJI USAGE BY USER:")
emoji_stats = df[df['sender'] != 'System'].groupby(
    'sender')['emoji_count'].sum().sort_values(ascending=False)

for sender, count in emoji_stats.items():
    print(f"    {sender:<12} : {count} emojis")

# ── Media sharing ──────────────────────────
print("\n[3] MEDIA SHARING BY USER:")
media_stats = df[df['sender'] != 'System'].groupby(
    'sender')['is_media'].sum().sort_values(ascending=False)

for sender, count in media_stats.items():
    print(f"    {sender:<12} : {int(count)} media messages")

CONTENT ANALYSIS

[1] TOP 15 MOST USED WORDS:
    morning         ███ (3)
    hey             ██ (2)
    everyone!       ██ (2)
    good            ██ (2)
    great!          ██ (2)
    project         ██ (2)
    same            ██ (2)
    morning!        █ (1)
    how             █ (1)
    are             █ (1)
    you?            █ (1)
    i'm             █ (1)
    doing           █ (1)
    just            █ (1)
    finished        █ (1)

[2] EMOJI USAGE BY USER:
    Alice        : 5 emojis
    Bob          : 4 emojis
    Dave         : 3 emojis
    Charlie      : 1 emojis

[3] MEDIA SHARING BY USER:
    Charlie      : 2 media messages
    Bob          : 1 media messages
    Alice        : 0 media messages
    Dave         : 0 media messages


In [8]:
# ============================================
# Cell 6: Test src/analyzer.py
# Verify all functions work correctly
# ============================================

from src.analyzer import (
    get_basic_stats,
    get_user_stats,
    get_time_stats,
    get_content_stats
)

# Test all 4 functions
basic   = get_basic_stats(df)
users   = get_user_stats(df)
time    = get_time_stats(df)
content = get_content_stats(df)

# Verify outputs
print("✅ get_basic_stats:")
print(f"   Total messages : {basic['total_messages']}")
print(f"   Duration       : {basic['duration_days']} days")

print("\n✅ get_user_stats:")
print(f"   Most active    : {users.iloc[0]['sender']}")

print("\n✅ get_time_stats:")
print(f"   Peak hour      : {time['peak_hour']}:00")
print(f"   Peak day       : {time['peak_day']}")

print("\n✅ get_content_stats:")
print(f"   Top word       : {content['top_words'][0]}")

print("\n🎉 analyzer.py is working perfectly!")

✅ get_basic_stats:
   Total messages : 25
   Duration       : 3 days

✅ get_user_stats:
   Most active    : Alice

✅ get_time_stats:
   Peak hour      : 9:00
   Peak day       : Tuesday

✅ get_content_stats:
   Top word       : ('morning', 3)

🎉 analyzer.py is working perfectly!
